# Práctica 4: Procesamiento del Lenguaje Natural

__Fecha de entrega: 16 de mayo de 2025__

El objetivo de esta práctica es aplicar los conceptos teóricos vistos en clase en el módulo de PLN.

Lo más importante en esta práctica no es el código Python, sino el análisis de los datos y modelos que construyas y las explicaciones razonadas de cada una de las decisiones que tomes. __No se valorarán trozos de código o gráficas sin ningún tipo de contexto o explicación__.

Finalmente, recuerda establecer el parámetro `random_state` en todas las funciones que tomen decisiones aleatorias para que los resultados sean reproducibles (los resultados no varíen entre ejecuciones).

In [ ]:
RANDOM_STATE = 1234

# 1) Carga del conjunto de datos

Los ficheros `fake.csv` y `true.csv` contienen artícuos de noticias clasificadas como fake (falsas) o true (reales) respectivamente. Cada noticia tiene como atributos:

*   Title: título de la noticia
*   Text: cuerpo del texto de la noticia
*   Subject: tema de la noticia
*   Date: fecha de publicación de la noticia

Muestra un ejemplo de cada clase.

Haz un estudio del conjunto de datos. ¿qué palabras aparecen más veces?, ¿tendría sentido normalizar de alguna manera el corpus?

Crea una partición de los datos dejando el 60% para entrenamiento, 20% para validación y el 20% restante para test. Comprueba que la distribución de los ejemplos en las particiones es similar.

In [ ]:
import pandas as pd
falso = pd.read_csv("fake.csv")
verdadero = pd.read_csv("true.csv")
#Cargamos las noticias falsas y las verdaderas, cada una por un lado.

In [ ]:
falso.head(1)
#Mostramos la primera noticia falsa para ver como es el formato.

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"


In [ ]:
verdadero.head(1)

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"


In [ ]:
#Para trabajar con un único dataframe, añadimos un campo tipo para diferenciar las noticias falsas de las verdaderas y concatenamos ambos dataframes en uno.
falso["tipo"] = "falsa"
verdadero["tipo"] = "verdadera"
datos = pd.concat([falso, verdadero])

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

#Inicializamos CountVectorizer, que transforma texto en otro formato para llevar el conteo de palabras.
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(datos["text"])
palabras = vectorizer.get_feature_names_out()

#Analizamos cuales son las palabras que más vecen aparecen en la columna text del dataframe, que es en la que aparece el contenido de las noticias.

frecuencias = np.asarray(X.sum(axis=0)).flatten()
#Sumamos las veces que aparece cada palabra en todos los textos

#Ahora asociamos cada palabra con su frecuencia y las ordenamos de mayor a menor
conteo_palabras = list(zip(palabras, frecuencias))
conteo_palabras = sorted(conteo_palabras, key=lambda x: x[1], reverse=True)

#Mostramos las 20 palabras que más veces aparecen
for palabra, freq in conteo_palabras[:20]:
    print(f"{palabra}: {freq}")

the: 1026019
to: 536553
of: 441915
and: 409052
in: 352815
that: 239899
on: 192185
for: 173375
is: 166728
trump: 134000
he: 133367
it: 132970
said: 132825
with: 117923
was: 115803
as: 105098
his: 96369
by: 95852
has: 88563
be: 83698


Como podemos observar, la mayoría de palabas que más aparecen no dan ningún tipo de información sobre el texto. Por tanto, podríamos normalizar el corpus eliminando palabras como "and", "a", etc. Cabe destacar que no estamos distinguiendo entre minúsculas y mayúsculas.

In [ ]:
#Eliminamos las palabras más típicas y que menos información nos aportan, y hacemos lo mismo que antes
vectorizer2 = CountVectorizer(stop_words='english')
X2 = vectorizer2.fit_transform(datos["text"])
palabras2 = vectorizer2.get_feature_names_out()

frecuencias2 = np.asarray(X2.sum(axis=0)).flatten()

conteo_palabras2 = list(zip(palabras2, frecuencias2))
conteo_palabras2 = sorted(conteo_palabras2, key=lambda x: x[1], reverse=True)

#Ahora mostramos las 20 palabras que más veces aparecen quitando palabras "inútiles".
for palabra, freq in conteo_palabras2[:20]:
    print(f"{palabra}: {freq}")

trump: 134000
said: 132825
president: 55892
people: 41857
state: 34488
new: 31311
reuters: 29425
clinton: 28695
obama: 28203
donald: 28127
government: 28048
house: 27753
states: 26843
republican: 25568
year: 24998
just: 24967
united: 23601
told: 23367
like: 22774
white: 22745


In [ ]:
from sklearn.model_selection import train_test_split
#Con esta línea, guardamos el 60% de los datos en entrenamiento, y dejamos el 40% restante en aux.
entrenamiento, aux = train_test_split(datos, test_size=0.4,random_state=RANDOM_STATE)
#Ahora, dividimos el 40% que nos queda en dos componentes iguales, para que tanto el test como la validación tengan un 20% de los datos totales.
validacion, test = train_test_split(aux, test_size=0.5,random_state=RANDOM_STATE)

conteoEntrenamiento = entrenamiento['tipo'].value_counts(normalize=True)
conteoValidacion = validacion['tipo'].value_counts(normalize=True)
conteoTest = test['tipo'].value_counts(normalize=True)

print("Textos total:",len(datos))
print("Textos entrenamiento:", len(entrenamiento))
print("Textos validacion:", len(validacion))
print("Textos test:", len(test))

#Con estas líneas, podemos ver si se ha hecho un reparto "justo", es decir, que las proporciones de noticias falsas y verdaderas sean similares
#tanto en entrenamiento como en validación y test.
print("Distribución de ejemplos en conjunto de entrenamiento:")
print(conteoEntrenamiento)
print("\nDistribución de ejemplos en conjunto de validación:")
print(conteoValidacion)
print("\nDistribución de ejemplos en conjunto de prueba:")
print(conteoTest)

Textos total: 44898
Textos entrenamiento: 26938
Textos validacion: 8980
Textos test: 8980
Distribución de ejemplos en conjunto de entrenamiento:
tipo
falsa        0.524501
verdadera    0.475499
Name: proportion, dtype: float64

Distribución de ejemplos en conjunto de validación:
tipo
falsa        0.518374
verdadera    0.481626
Name: proportion, dtype: float64

Distribución de ejemplos en conjunto de prueba:
tipo
falsa        0.523051
verdadera    0.476949
Name: proportion, dtype: float64


# 2) Representación como bolsa de palabras

Elige justificadamente una representación de bolsa de palabras y aplícala.
Muestra un ejemplo antes y después de aplicar la representación. Explica los cambios.

In [ ]:
v = CountVectorizer(stop_words='english')
entrenamientoVector =v.fit_transform(entrenamiento["text"])
feature_names = v.get_feature_names_out()
print(feature_names[:100])
print(feature_names[-100:])

['00' '000' '0000' '00004' '000048' '000063' '00007' '000270' '00042'
 '0009' '000938' '000a' '000after' '000although' '000american'
 '000california' '000cases' '000cylvia' '000ecuador' '000florida'
 '000georgia' '000illegal' '000illinois' '000in' '000jose' '000kyrgyzstan'
 '000m' '000michigan' '000new' '000oman' '000s' '000saudi' '000south'
 '000th' '000that' '000the' '000uterine' '001' '00106' '0011' '00155'
 '0018' '00193' '001romney' '002' '0020' '002singapore' '003' '004' '0047'
 '004saint' '005' '0050' '005380' '005930' '006' '00654' '00684' '007'
 '0075' '007kzman' '008' '00867' '00am' '00c6j7capuhttps' '00hex' '00o'
 '00pm' '00pme' '00yecahb4d' '01' '010' '0100' '01000110' '01010101'
 '0109' '011' '0112' '01233' '0129' '013' '0130' '014' '015' '01517'
 '015760' '016' '017' '019' '01am' '01vcq3uzgx' '01wcckwopi' '01wiw6prwy'
 '02' '020' '0200' '020malta' '021' '0211' '02197']
['zucchiniblute' 'zucker' 'zuckerberg' 'zuckerman' 'zuckermann'
 'zuckersaid' 'zuckett' 'zuckoff' 'zuffa

Hacemos la respresentacion de bolsa de las palabras eliminando las stop_words del inglés para posteriormente transformarlo en un conjunto de entrenamiento del que imprimimos las primeras y las últimas 100 palabras del vocabulario generado. No obstante viendo las palabras imprimidas es evidente que hay mucho ruido puesto que la mayoría de ellas son palabras que no existen, así que procedemos a arreglar esto más adelante.


In [ ]:
import numpy as np
import numpy.ma as ma

def write_terms (feature_names, data, vector_data, index):
    '''
    Escribe los términos presentes en un mensaje representado como bolsa de palabras.

    - feature_names: terminos usados para vectorizar
    - data: lista de mensajes original (si data==None no se muestra el mensaje original)
    - vector_data: matriz (dispersa) de mensaje vectorizados
    - index: posición del mensaje a mostrar
    '''
    # máscara para seleccionar sólo el mensaje en posición index
    mask=vector_data[index,:]>0

    # términos que aparecen en ese mensaje vectorizado
    terminos = ma.array(feature_names, mask = ~(mask[0].toarray()))

    # mostrar mensaje original
    if data is not None:
        print('Mensaje', index, ':', data[index])

    # mostrar términos que aparecen en el mensaje vectorizado
    print('Mensaje', index, 'vectorizado:', terminos.compressed(),'\n')

In [ ]:
write_terms(feature_names, None, entrenamientoVector, 10)
write_terms(feature_names, None, entrenamientoVector, 1)

Mensaje 10 vectorizado: ['2017' '24' '9ekkg0nr5i' 'com' 'does' 'fox' 'foxnews' 'gop' 'march'
 'news' 'old' 'people' 'pic' 'rosadelauro' 'stand' 'twitter'] 

Mensaje 1 vectorizado: ['14' '2016that' 'able' 'according' 'aides' 'alliance' 'alliances'
 'america' 'aren' 'babysit' 'bashing' 'bender' 'called' 'campaign'
 'change' 'chief' 'com' 'comes' 'commitment' 'committed' 'community'
 'conservatives' 'continued' 'core' 'cornerstone' 'country' 'deliver'
 'didn' 'donald' 'easy' 'elected' 'embracing' 'entire' 'europe'
 'experience' 'experienced' 'expressed' 'fact' 'fully' 'functions' 'getty'
 'global' 'going' 'good' 'great' 'guide' 'hands' 'hate' 'heart' 'hill'
 'himhttps' 'hire' 'hired' 'hold' 'house' 'idea' 'ii' 'important' 'job'
 'just' 'know' 'lead' 'leader' 'learn' 'let' 'little' 'maintaining' 'man'
 'mark' 'meeting' 'messages' 'mexico' 'michael' 'michaelcbender' 'monday'
 'months' 'nato' 'needed' 'new' 'november' 'obama' 'obamacare' 'obsolete'
 'office' 'organizer' 'oval' 'pay' 'photo' 

Como podemos observar, hay palabras extrañas como x4edzsf8uy, zltpsqswge y 9ekkg0nr5i. Por tanto, nos vamos a restringir a palabras de un diccionario para evitar este problema. Y es que aunque el ruido sea menor, esto es debido simplemente a que estamos viendo mensajes concretos y la realidad es que el ruido es relativamente poco frecuente por lo que al tomar las palabras del mensaje parecen más legibles, lo que no significa que haya desapareido este ruido.



In [ ]:
with open('words.txt') as f:
    dictionary = f.read().splitlines()

v = CountVectorizer(vocabulary=dictionary, stop_words='english')

fNames = v.get_feature_names_out()

print(len(fNames))
print(fNames[:100])
entrenamientoVector = v.fit_transform(entrenamiento["text"])
print(entrenamientoVector.shape[1])

466550
['2' '1080' '&c' '10-point' '10th' '11-point' '12-point' '16-point'
 '18-point' '1st' '2,4,5-t' '2,4-d' '20-point' '2D' '2nd' '30-30' '3D'
 '3-D' '3M' '3rd' '48-point' '4-D' '4GL' '4H' '4th' '5-point' '5-T' '5th'
 '6-point' '6th' '7-point' '7th' '8-point' '8th' '9-point' '9th' 'a' "a'"
 'a-' 'A&M' 'A&P' 'A.' 'A.A.A.' 'A.B.' 'A.B.A.' 'A.C.' 'A.D.' 'A.D.C.'
 'A.F.' 'A.F.A.M.' 'A.G.' 'A.H.' 'A.I.' 'A.I.A.' 'A.I.D.' 'A.L.' 'A.L.P.'
 'A.M.' 'A.M.A.' 'A.M.D.G.' 'A.N.' 'a.p.' 'a.r.' 'A.R.C.S.' 'A.U.'
 'A.U.C.' 'A.V.' 'a.w.' 'A.W.O.L.' 'A/C' 'A/F' 'A/O' 'A/P' 'A/V' 'A1'
 'A-1' 'A4' 'A5' 'AA' 'AAA' 'AAAA' 'AAAAAA' 'AAAL' 'AAAS' 'Aaberg'
 'Aachen' 'AAE' 'AAEE' 'AAF' 'AAG' 'aah' 'aahed' 'aahing' 'aahs' 'AAII'
 'aal' 'Aalborg' 'Aalesund' 'aalii' 'aaliis']


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:1368: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


466550


Ahora cargamos el diccionario personalizado de palabras words.txt para decir cuántas palabras distintas hay (466550) y mostrar las primeras 100 de ellas presentes en la bolsa de palabras.



In [ ]:
print(entrenamientoVector[10])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4 stored elements and shape (1, 466550)>
  Coords	Values
  (0, 111569)	2
  (0, 289704)	1
  (0, 381893)	1
  (0, 421507)	1


In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

# Calculamos el valor TF-IDF

tfidfer = TfidfTransformer()
train_preprocessed = tfidfer.fit_transform(entrenamientoVector)

print(train_preprocessed[10])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (1, 466550)>
  Coords	Values
  (0, 111569)	0.7556351125505866
  (0, 289704)	0.2378452470922864
  (0, 381893)	0.4922072774696405
  (0, 421507)	0.3608007914655896


Ahora al utilizar TF-IDF vemos como los valores pasan de ser el número de veces que aparece cada palabra a la frecuencia con la que aparecen multiplicada por la importancia que tiene esa palabra (si aparece pocas veces tendrá una importancia mayor y viceversa)



In [ ]:
write_terms(fNames, None, entrenamientoVector, 10)
write_terms(fNames, None, entrenamientoVector, 1)

Mensaje 10 vectorizado: ['does' 'people' 'stand' 'twitter'] 

Mensaje 1 vectorizado: ['able' 'according' 'alliances' 'aren' 'babysit' 'bashing' 'called'
 'campaign' 'change' 'chief' 'comes' 'commitment' 'committed' 'community'
 'conservatives' 'continued' 'cornerstone' 'country' 'deliver' 'didn'
 'easy' 'elected' 'embracing' 'entire' 'experience' 'experienced'
 'expressed' 'fact' 'fully' 'functions' 'global' 'hate' 'heart' 'hire'
 'hired' 'hold' 'years' 'ii' 'important' 'know' 'leader' 'learn' 'let'
 'maintaining' 'meeting' 'messages' 'months' 'needed' 'obsolete' 'office'
 'organizer' 'pay' 'photo' 'played' 'plans' 'political' 'president'
 'process' 'recognition' 'record' 'referred' 'relationship'
 'relationships' 'reportedly' 'resolve' 'right' 'robust' 'said' 'scope'
 'score' 'security' 'serve' 'spend' 'spent' 'staff' 'stick' 'strategic'
 'suckers' 'surprised' 'taking' 'team' 'think' 'thought' 'told' 'trail'
 'transatlantic' 'trump' 'twitter' 'unaware' 'vital' 'voters' 'wasn'
 'weaken

Así pues tras aplicar TF-IDF podemos ver como ahora las palabras tienden a ser más legibles e informativas puesto que se han eliminado las que tienen poco valor iinformativo.


In [ ]:
# Tomamos los textos del conjunto de test y los transformamos en una matriz
# de palabras. Al usar "transform" toma como referencia únicamente las palabras
# encontradas en el conjunto de entrenamiento
testVector=v.transform(test["text"])
# Calculamos el valor TF-IDF
# Al usar "transform" toma como IDF el del conjunto de entrenamiento
test_preprocessed=tfidfer.transform(testVector)

# 3) Aplica 3 algoritmos de aprendizaje automático para resolver la tarea

Justifica porqué los has elegido.
Ajusta los modelos respecto a un hiperparámetro que consideres oportuno. Justifica tu elección.
Explica los resultados obtenidos.

In [ ]:
#Primero utilizamos un árbol de decisión y evaluamos su desempeño según el hiperparámetro max_depth
#Este algoritmo es bastante sencillo tanto de interpretar como de visualizar. Además, es de los algoritmos más rápidos y eficientes.
from sklearn import tree
import numpy as np
from sklearn.metrics import precision_score


for depth in range(1, 25):  # Probamos con profundidad máxima del árbol desde 1 hasta 24.
#A primera vista, se elegirá un valor medio-alto, ya que un número demasiado bajo puede causar subajuste, pero un número muy elevado puede
#causar sobreajuste.
    tree_classifier = tree.DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    tree_classifier.fit(train_preprocessed, entrenamiento["tipo"])
    #Se predice sobre el conjunto de entrenamiento y sobre el de test.
    tree_train_predictions = tree_classifier.predict(train_preprocessed)
    tree_test_predictions = tree_classifier.predict(test_preprocessed)
    #Ahora vemos la precisión para la pos_label verdadera, es decir, las noticias verdaderas (precision=VP/(VP+FP))
    precTrain = precision_score(entrenamiento["tipo"],tree_train_predictions, pos_label='verdadera', average='binary', zero_division=1)
    precTest = precision_score(test["tipo"],tree_test_predictions, pos_label='verdadera', average='binary', zero_division=1)
    #Lo imprimimos para que sea fácil de visualizar y de hacer una buena elección del hiperparámetro.
    print(f"max_depth={depth}: precision entrenamiento={precTrain}, precision test={precTest}")


max_depth=1: precision entrenamiento=0.7751766784452296, precision test=0.770132407206425
max_depth=2: precision entrenamiento=0.8324732342293769, precision test=0.8217194570135746
max_depth=3: precision entrenamiento=0.8604707595243872, precision test=0.8477434679334916
max_depth=4: precision entrenamiento=0.8746781828751765, precision test=0.8666010337189269
max_depth=5: precision entrenamiento=0.8766175632782616, precision test=0.8665696263949539
max_depth=6: precision entrenamiento=0.8412732474964235, precision test=0.8357727077055531
max_depth=7: precision entrenamiento=0.8594508440194161, precision test=0.8463711429813124
max_depth=8: precision entrenamiento=0.8607413768427079, precision test=0.8442817144087874
max_depth=9: precision entrenamiento=0.8730112814579115, precision test=0.8496647198788665
max_depth=10: precision entrenamiento=0.8885544380001469, precision test=0.8626977152899824
max_depth=11: precision entrenamiento=0.9007963086998586, precision test=0.870651204281891

Como podemos observar, la precisión va subiendo conforme aumenta la profundidad máxima. Sin embargo, este aumento es bastante gradual y el coste también va en aumento. Por tanto, elegir la profundidad máxima, a pesar de tener el mayor valor de precisión, no es una buena idea porque no compensa la mejora en precisión con el aumento del coste. Por tanto, consideramos que es una buena opción elegir como profundidad máxima 19, ya que tiene una precisión lo suficientemente buena en comparación con las siguientes como para parar ahí.

In [ ]:
#K-NN es un modelo sencillo de entender y para problemas donde el espacio de datos está bien organizado como es nuestro caso
#funciona bastante bien. Además, se adapta bastante bien si hay palabras que aparecen más comúnmente en las noticias falsas que en
#las verdaderas, y viceversa.
from sklearn import neighbors
from sklearn.metrics import precision_score
import numpy as np

for k in range(1, 25):  # Probamos con número de vecinos desde 1 hasta 24.
    knn_classifier = neighbors.KNeighborsClassifier(n_neighbors=k)
    knn_classifier.fit(train_preprocessed, entrenamiento["tipo"])

    #Hacemos al igual que antes, el predict tanto del entrenamiento como del test.
    knn_train_predictions = knn_classifier.predict(train_preprocessed)
    knn_test_predictions = knn_classifier.predict(test_preprocessed)

    precisionTrain = precision_score(entrenamiento["tipo"], knn_train_predictions, average='weighted', zero_division=0)
    precisionTest = precision_score(test["tipo"], knn_test_predictions, average='weighted', zero_division=0)
    #Aquí, el hiperparámetro k si es demasiado pequeño puede causar sobreajuste, y si es demasiado grande subajuste.
    print(f"k={k}: precisión en entrenamiento = {precisionTrain}, precisión en test = {precisionTest}")

k=1: precisión en entrenamiento = 0.9999628803464031, precisión en test = 0.759789881874856
k=2: precisión en entrenamiento = 0.7858158838275967, precisión en test = 0.7401952736477115
k=3: precisión en entrenamiento = 0.7807209382787395, precisión en test = 0.7331082540325302
k=4: precisión en entrenamiento = 0.7598821834728152, precisión en test = 0.73717179109451
k=5: precisión en entrenamiento = 0.7563942948080858, precisión en test = 0.7282262572697378
k=6: precisión en entrenamiento = 0.7511580638231864, precisión en test = 0.7265721584409398
k=7: precisión en entrenamiento = 0.7461063237993798, precisión en test = 0.7270363044458192
k=8: precisión en entrenamiento = 0.7472429198777066, precisión en test = 0.7337635346907583
k=9: precisión en entrenamiento = 0.7417737087158907, precisión en test = 0.734199266194971
k=10: precisión en entrenamiento = 0.7452646142858852, precisión en test = 0.7423389936672475
k=11: precisión en entrenamiento = 0.7445273438625563, precisión en test 

Aquí podemos ver que en k=1 hay un claro sobreajuste, ya que la precisión de entrenamiento tiene un valor muy mayor a la precisión en el test.
En cuanto al resto, podemos ver que son muy similares todos los valores de precisión, así que lo mejor sería elegir un valor de k no demasiado bajo para evitar el sobreajuste, pero tampoco elegir uno demasiado alto, ya que la mejora de precisión es muy escasa y así reducimos el número de comparaciones a hacer. Por tanto, un buen valor de k sería k=12 o alguna similar.

In [ ]:
#Ahora utilizamos Naive Bayes Multinomial, que suele funcionar bien para problemas de clasificación de texto.
#Este es un algoritmo bastante eficiente, ya que tiene complejidad O(n), por lo que puede trabajar con un gran volumen de datos con facilidad,
#como es en este caso. Además, es muy eficaz en cuanto a problemas de clasificación de texto, y maneja palabras raras que no suelen aparecer
#con el suavizado (alpha).
from sklearn.naive_bayes import MultinomialNB
import numpy as np

#Cuando alpha>1 reduce el peso de las probabilidades de las características observadas y hace que el modelo sea más conservador.
#Cuando alpha<1 permite un mayor peso a las características observadas
for a in [0.01, 0.1, 0.5, 1.0, 1.5, 2.0, 5.0]:#Probamos con estos valores de alpha
    mnb_classifier = MultinomialNB(alpha=a)
    mnb_classifier.fit(train_preprocessed, entrenamiento["tipo"])

    mnb_train_predictions = mnb_classifier.predict(train_preprocessed)
    mnb_test_predictions = mnb_classifier.predict(test_preprocessed)

    prec_train = precision_score(entrenamiento["tipo"], mnb_train_predictions, average='macro', zero_division=0)
    prec_test = precision_score(test["tipo"], mnb_test_predictions, average='macro', zero_division=0)

    print(f"alpha={a} → precisión entrenamiento: {prec_train}, test: {prec_test}")

alpha=0.01 → precisión entrenamiento: 0.9508296755760096, test: 0.924955446632139
alpha=0.1 → precisión entrenamiento: 0.9445765425034942, test: 0.9244836935932006
alpha=0.5 → precisión entrenamiento: 0.9386046932313111, test: 0.9231626099281283
alpha=1.0 → precisión entrenamiento: 0.9351122852812359, test: 0.9235823516010945
alpha=1.5 → precisión entrenamiento: 0.933435068215946, test: 0.9235823516010945
alpha=2.0 → precisión entrenamiento: 0.9325517129961127, test: 0.9224915022926272
alpha=5.0 → precisión entrenamiento: 0.9296002337754072, test: 0.9214977765320593


Aunque parezca que la mejor es alpha=0.01, si queremos generalizar este problema lo mejor es tomar un alpha algo más "medio". En este caso justo funciona, pero probando en otro caso si tomáramos ese alpha tan pequeño podríamos estar provocando sobreajuste. Por tanto, con la poca diferencia de precisión que hay, lo mejor sería tomar, por ejemplo, alpha=1.

# 4) Construye redes neuronales con Keras con distintas maneras de usar word embeddings

Justifica tus decisiones y explica los resultados obtenidos.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing import sequence
#Tokenizer convertirá los textos en cadenas de enteros.

max_words = 1500 #número máximo de palabras que se considerarán
max_comment_length = 20 #longitud máxima de las cadenas de texto

pd.set_option('future.no_silent_downcasting', True)
datos["tipo"] = datos["tipo"].replace({"verdadera": 1, "falsa": 0}) #convertimos las etiquetas en valores numéricos
tokenizer = Tokenizer(num_words=max_words) #Creamos el objeto tokenizer que tomará 1500 palabras
tokenizer.fit_on_texts(datos["text"]) #Aquí lo aplicamos a nuestras noticias
sequences = tokenizer.texts_to_sequences(datos["text"]) #Convertimos los textos en cadenas de enteros.

word_index = tokenizer.word_index #Es un diccionario donde las claves son las palabras y los valores son los índices asociados a cada palabra
print('Se han encontrado %s tokens únicos.' % len(word_index)) #Mostramos cuantas palabras hay
max_words = len(word_index)

data = sequence.pad_sequences(sequences, maxlen=max_comment_length)

Se han encontrado 138021 tokens únicos.


In [ ]:
print(datos.text[4])
print(data[4])
print(datos.tipo[4])

4    Pope Francis used his annual Christmas Day mes...
4    SEATTLE/WASHINGTON (Reuters) - President Donal...
Name: text, dtype: object
[   3   29   91   24    2  209  423   32   25   41 1174  685  550   41
    2    7 1030   20  567  501]
4    0
4    1
Name: tipo, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

#valor de random state
rs=333

d=datos.values

#Dividimos los datos para test y entrenamiento. El 80% serán de entrenamiento y el 20% de test.
x_train, x_test, y_train, y_test = train_test_split(data, datos.tipo, test_size=0.20, random_state=rs, stratify = datos.tipo)

print("Training texts:", len(y_train))
print("Test texts:", len(y_test))

Training texts: 35918
Test texts: 8980


In [ ]:
# Fijamos el tamaño de los embedding a 50 dimensiones
embedding_dim = 50

In [ ]:
#SIN EMBEDDINGS PRE-ENTRENADOS

from keras.models import Sequential
from keras.layers import Flatten, Dense, Embedding

model1 = Sequential() #Creamos el modelo secuencial

model1.add(Embedding(max_words, embedding_dim, input_length=max_comment_length)) # Convertimos convierte las palabras de las secuencias de texto en vectores densos
#es decir, en embeddings.

model1.add(Flatten()) #Pasamos el tensor de 3D a 2D (max_words, max_comment_length * embedding_dim).

# We add the classifier on top
model1.add(Dense(1, activation='sigmoid')) #Capa de una sola neurona, que produce una salida entre 0 y 1, perfecto para nuestro caso de clasificación.

model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model1.summary() #Imprimimos un resumen del modelo.

#Entrenamos el modelo con los datos de entrenamiento.
history = model1.fit(x_train, y_train,
                    epochs=20,
                    batch_size=32,
                    validation_data=(x_test, y_test))

#Evaluamos el rendimiento del modelo con el conjunto de pruebas.
score1 = model1.evaluate(x_test, y_test)

print("Accuracy: %.2f%%" % (score1[1]*100))

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.8921 - loss: 0.3173 - val_accuracy: 0.9601 - val_loss: 0.1067
Epoch 2/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9694 - loss: 0.0870 - val_accuracy: 0.9655 - val_loss: 0.0929
Epoch 3/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9840 - loss: 0.0556 - val_accuracy: 0.9688 - val_loss: 0.0870
Epoch 4/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9935 - loss: 0.0313 - val_accuracy: 0.9716 - val_loss: 0.0839
Epoch 5/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9980 - loss: 0.0161 - val_accuracy: 0.9735 - val_loss: 0.0843
Epoch 6/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9996 - loss: 0.0079 - val_accuracy: 0.9729 - val_loss: 0.0917
Epoch 7/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 1.0000 - loss: 0.0044 - val_accuracy: 0.9731 - val_loss: 0.0983
Epoch 8/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9999 -

Aquí se puede apreciar que la precisión es prácticamente perfecta desde el principio, por lo que no es necesario realizar tantos "epochs" al entrenarlo, ya que a partir de 7 o 8 empieza a ser perfecto.

In [ ]:
#CON EMBDDINGS PRE-ENTRENADOS.
import os
import numpy as np

glove_dir = './glove'

embeddings_index = {} #Creamos un diccionario vacío.
f = open(os.path.join(glove_dir, 'glove.6B.50d.txt')) #Abrimos el archivo para acceder a los embeddings preentrenados.
#Ahora, leemos todas las líneas del archivo.
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()
#Cerramos el archivo y mostramos cuantos vectores se han encontrado.

print('Found %s word vectors.' % len(embeddings_index))

Found 400000 word vectors.


In [ ]:
embedding_dim = 50

embedding_matrix = np.zeros((max_words, embedding_dim)) #Creamos una matriz de ceros de tamaño (max_words, embedding_dim).
#Es decir, tendrá una fila para cada palabra en el vocabulario (max_words), y cada fila será un vector de dimensión embedding_dim.
#Ahora, recorremos el diccionario.
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word) #Se busca el vector de embeddings correspondiente a la palabra word en el diccionario embeddings_index
    if i < max_words:
        if embedding_vector is not None:#Solo se asignan los vectores de embeddings que realmente existen
            embedding_matrix[i] = embedding_vector

In [ ]:
#EMBEDDINGS PRE-ENTRENADOS CONGELADOS

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense

import numpy as np

# Debugging prints
print(f"max_words: {max_words}, embedding_dim: {embedding_dim}, max_comment_length: {max_comment_length}")
print(f"embedding_matrix shape: {embedding_matrix.shape}")

model2 = Sequential() #Creamos el nuevo modelo secuencial y lo configuramos.
model2.add(Embedding(max_words, embedding_dim, input_length=max_comment_length))
# model2.add(Embedding(max_words, embedding_dim, input_length=max_comment_length, weights=embedding_matrix))

print("Layer weights before setting:", model2.layers[0].weights)

model2.build(input_shape=(None, max_comment_length)) #Creamos el modelo

model2.add(Flatten()) #Lo aplanamos a 1D
model2.add(Dense(1, activation='sigmoid'))#Se añade la capa densa que da una salida que será o 0 o 1.
model2.summary()

max_words: 138021, embedding_dim: 50, max_comment_length: 20
embedding_matrix shape: (138021, 50)
Layer weights before setting: []


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 20, 50)         │     6,901,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         1,001 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,902,051 (26.33 MB)

 Trainable params: 6,902,051 (26.33 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Verificamos que la matriz tenga las dimensiones adecuadas.
if embedding_matrix.shape == (max_words, embedding_dim):
    model2.layers[0].set_weights([embedding_matrix]) #Establecemos los pesos de la primera capa o embedding
    model2.layers[0].trainable = False #La configuramos para que no se entrene
else:
    print("Error: embedding_matrix has incorrect shape.")

In [ ]:
#Ahora, compiilamos entrenamos y evaluamos los conjuntos de prueba.
model2.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model2.fit(x_train, y_train,
                    epochs=20,
                    batch_size=32,
                    validation_data=(x_test, y_test))

score2 = model2.evaluate(x_test, y_test)

Epoch 1/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 1s 378us/step - accuracy: 0.8256 - loss: 0.3854 - val_accuracy: 0.8982 - val_loss: 0.2605
Epoch 2/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 0s 339us/step - accuracy: 0.9047 - loss: 0.2502 - val_accuracy: 0.9030 - val_loss: 0.2474
Epoch 3/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 0s 339us/step - accuracy: 0.9078 - loss: 0.2407 - val_accuracy: 0.9071 - val_loss: 0.2407
Epoch 4/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 0s 340us/step - accuracy: 0.9055 - loss: 0.2428 - val_accuracy: 0.9033 - val_loss: 0.2391
Epoch 5/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 0s 338us/step - accuracy: 0.9118 - loss: 0.2294 - val_accuracy: 0.9065 - val_loss: 0.2353
Epoch 6/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 0s 339us/step - accuracy: 0.9144 - loss: 0.2213 - val_accuracy: 0.9007 - val_loss: 0.2472
Epoch 7/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 0s 347us/step - accuracy: 0.9115 - loss: 0.2295 - val_accuracy: 0.9060 - val_loss: 0.2336
Epoch 8/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 0s 339us/step - accuracy: 0.9115 -

Aquí la precisión es algo menor que en el modelo anterior, pero la variación de "epochs" prácticamente no varía la precisión, llegando incluso a empeorarla en algunos casos.

In [ ]:
#EMBEDDINGS PREENTRENADOS SIN CONGELAR

from keras.models import Sequential
from keras.layers import Embedding, Flatten, Dense

model3 = Sequential()#Creamos el nuevo modelo secuencial.
model3.add(Embedding(max_words, embedding_dim, input_length=max_comment_length))
model3.add(Flatten()) #Aplanamos de 3D a 2D.
model3.add(Dense(1, activation='sigmoid')) #Salida 1 o 0.
model3.summary()

model3.build(input_shape=(None, max_comment_length)) #Nos aseguramos de que esté correctamente inicializado.
model3.layers[0].set_weights([embedding_matrix]) #Se asignan los pesos preentrenados al primer embedding.
model3.layers[0].trainable = True #Para que sean actualizables durante el entrenamiento

#Compilamos entrenamos y evaluamos.
model3.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
history = model3.fit(x_train, y_train,
                    epochs=20,
                    batch_size=32,
                    validation_data=(x_test, y_test))

score3 = model3.evaluate(x_test, y_test)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.8669 - loss: 0.3092 - val_accuracy: 0.9501 - val_loss: 0.1348
Epoch 2/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9554 - loss: 0.1212 - val_accuracy: 0.9590 - val_loss: 0.1111
Epoch 3/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9691 - loss: 0.0874 - val_accuracy: 0.9605 - val_loss: 0.1041
Epoch 4/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9780 - loss: 0.0674 - val_accuracy: 0.9635 - val_loss: 0.0983
Epoch 5/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9863 - loss: 0.0486 - val_accuracy: 0.9665 - val_loss: 0.0938
Epoch 6/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9917 - loss: 0.0330 - val_accuracy: 0.9689 - val_loss: 0.0943
Epoch 7/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9964 - loss: 0.0217 - val_accuracy: 0.9710 - val_loss: 0.0952
Epoch 8/20
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9983 -

En este caso, sí que se aprecia algo más el crecimiento en la precisión al principio, pero una vez llegado a los 8 epochs empieza a ser casi perfecta, por lo que no merece la pena utilizar más.

# 5) Aplica los modelos construidos a los datos de test y compáralos.

Calcula las métricas de recall, precisión y f1.
Discute cual es el mejor modelo y cual es peor y porqué.

In [ ]:
# Para todos los modelos, usamos .predict() y redondeamos para convertir probabilidades en 0 o 1
y_pred1 = (model1.predict(x_test) > 0.5).astype("int32")
y_pred2 = (model2.predict(x_test) > 0.5).astype("int32")
y_pred3 = (model3.predict(x_test) > 0.5).astype("int32")


281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 236us/step
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 206us/step
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 203us/step


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def eval_model(y_test, y_pred, name="Modelo"):
    print(name)
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:   ", recall_score(y_test, y_pred))
    print("F1 Score: ", f1_score(y_test, y_pred))
    print()

In [ ]:
eval_model(y_test, y_pred1, "Modelo 1 - Embedding aleatorio")
eval_model(y_test, y_pred2, "Modelo 2 - GloVe congelado")
eval_model(y_test, y_pred3, "Modelo 3 - GloVe entrenable")

Modelo 1 - Embedding aleatorio
Precision: 0.9713950762016412
Recall:    0.9670868347338936
F1 Score:  0.9692361679728623

Modelo 2 - GloVe congelado
Precision: 0.8719393282773564
Recall:    0.9393090569561158
F1 Score:  0.9043712776716485

Modelo 3 - GloVe entrenable
Precision: 0.9691155825924193
Recall:    0.9668534080298786
F1 Score:  0.9679831736387007



En primer lugar para valorar el modelo evidentemente nos fijamos en las características expuestas: Precision (proporcion de valores positivos correctos), recall (proporción de casos positivos correctamente analizados) y F1 ( valor armónico entre ambos). Podemos observar como el embedding aleatorio es el que tiene mejores parámetros y por tanto el mejor modelo. El peor modelo resulta el GloVe Congelado y es que aunque los embeddings estén preentrenados estos no se ajustan ni entrenan durante el proceso de aprendizaje lo que provoca que sean poco eficaces. Es por esto que el modelo 3 donde estos si se ajustan durante dicho proceso ya tiene muchos mejores valores siendo más similar al modelo aleatorio y con una mayor similitud entre los valores de Precision y Recall.